<a href="https://colab.research.google.com/github/fbrum44/tcc_wikiart/blob/main/notebooks/07_shortcut_learning_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Etapa 10 — Análise de Shortcut Learning

Este notebook investiga a presença de **shortcut learning** nos modelos ResNet-50 e ViT-B/16 utilizados para classificação de movimentos artísticos.

O objetivo é verificar se características relacionadas ao **gênero temático das pinturas** — como `religious_painting`, `landscape`, `portrait` e `still_life` — podem funcionar como pistas correlacionadas às macroclasses, facilitando a classificação sem necessariamente representar características estilísticas próprias dos movimentos artísticos.

A análise utiliza o conjunto de teste previamente congelado, com **1.500 pinturas**, distribuídas igualmente entre:

- Renaissance;
- Baroque;
- Impressionism;
- Cubism.

Os principais experimentos desta etapa são:

1. análise da associação entre gênero e macroclasse;
2. avaliação dos modelos por gênero;
3. construção de um baseline utilizando apenas `genre`;
4. avaliação controlada no subconjunto `portrait`;
5. análise das confusões temáticas entre Renaissance e Baroque;
6. consolidação das evidências de shortcut learning.

> Importante: a presença de correlação entre gênero e classe demonstra a existência de uma **pista potencialmente explorável** pelos modelos, mas não prova, isoladamente, que essa pista seja causalmente responsável por suas decisões.

## 0. Preparação do ambiente

As análises desta etapa utilizam:

- os metadados dos conjuntos de treino e teste;
- as predições por pintura produzidas na Etapa 9;
- os resultados previamente congelados da ResNet-50 e do ViT-B/16.

Nenhum modelo será treinado novamente nesta etapa.

Os arquivos grandes contendo as predições permanecem armazenados no Google Drive compartilhado, enquanto códigos e resultados compactos são mantidos no repositório GitHub.

In [ ]:
from pathlib import Path
import subprocess
import os

REPO_DIR = Path("/content/tcc_wikiart")

if not REPO_DIR.exists():
    print("Repositório não encontrado. Clonando...")

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/fbrum44/tcc_wikiart.git",
            str(REPO_DIR),
        ],
        check=True,
    )

else:
    print("Repositório já existe.")

os.chdir(REPO_DIR)

print()
print("Diretório atual:", Path.cwd())

subprocess.run(
    ["git", "log", "--oneline", "-5"],
    cwd=REPO_DIR,
)

Repositório já existe.

Diretório atual: /content/tcc_wikiart


CompletedProcess(args=['git', 'log', '--oneline', '-5'], returncode=0)

In [ ]:
from pathlib import Path

arquivos = [
    Path(
        "/content/tcc_wikiart/"
        "src/evaluation/shortcut_analysis.py"
    ),
    Path(
        "/content/tcc_wikiart/"
        "notebooks/07_shortcut_learning_colab.ipynb"
    ),
]

for arquivo in arquivos:
    print(
        arquivo.name,
        "→",
        arquivo.exists(),
    )

shortcut_analysis.py → True
07_shortcut_learning_colab.ipynb → True


## 10A — Distribuição de gênero por macroclasse

O primeiro experimento avalia a associação entre o gênero temático (`Genero`) e a macroclasse artística (`Macro_Estilo`).

O objetivo é identificar se determinados conteúdos aparecem com frequências muito diferentes entre Renaissance, Baroque, Impressionism e Cubism.

Uma associação forte entre gênero e classe representa uma possível oportunidade de **shortcut learning**, pois características temáticas da imagem podem fornecer informações sobre o rótulo mesmo sem representar diretamente o estilo artístico.

Nesta etapa também são integradas às informações de gênero as predições finais por pintura da ResNet-50 e do ViT-B/16.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

DRIVE_BASE = Path(
    "/content/drive/MyDrive/"
    "TCC_WikiArt_Compartilhado/"
    "outputs/evaluation"
)

arquivos_drive = [
    DRIVE_BASE
    / "resnet50"
    / "test_painting_predictions.csv.gz",

    DRIVE_BASE
    / "vit_b16"
    / "test_painting_predictions.csv.gz",
]

for arquivo in arquivos_drive:
    print(
        arquivo,
        "→",
        arquivo.exists(),
    )

/content/drive/MyDrive/TCC_WikiArt_Compartilhado/outputs/evaluation/resnet50/test_painting_predictions.csv.gz → True
/content/drive/MyDrive/TCC_WikiArt_Compartilhado/outputs/evaluation/vit_b16/test_painting_predictions.csv.gz → True


In [ ]:
!python -m src.evaluation.shortcut_analysis \
  --resnet-predictions "/content/drive/MyDrive/TCC_WikiArt_Compartilhado/outputs/evaluation/resnet50/test_painting_predictions.csv.gz" \
  --vit-predictions "/content/drive/MyDrive/TCC_WikiArt_Compartilhado/outputs/evaluation/vit_b16/test_painting_predictions.csv.gz"


=== Etapa 10A — preparação concluída ===
Treino: 7000 pinturas
Teste: 1500 pinturas
Base integrada: 1500 pinturas
Gêneros no treino: 11
Gêneros no teste: 11

Unknown Genre no treino: 1147
Unknown Genre no teste: 261

Distribuição de gênero por macroclasse no teste:
Genero         Unknown Genre  abstract_painting  cityscape  genre_painting  illustration  landscape  nude_painting  portrait  religious_painting  sketch_and_study  still_life
Macro_Estilo                                                                                                                                                                  
Renaissance            14.40                0.0       0.00            2.40          0.80       0.80           0.80     18.13               56.80              5.87        0.00
Baroque                21.33                0.0       1.60           17.07          0.53       2.93           0.80     22.40               25.07              4.80        3.47
Impressionism          13.60     

### Resultado — associação entre gênero e macroclasse

A distribuição revelou diferenças temáticas importantes entre as quatro macroclasses.

Entre os principais padrões observados:

- **Renaissance:** 56,80% das pinturas são `religious_painting`;
- **Baroque:** distribuição mais heterogênea, com 25,07% `religious_painting`, 22,40% `portrait` e 17,07% `genre_painting`;
- **Impressionism:** 33,87% das pinturas são `landscape`;
- **Cubism:** 26,40% das pinturas são `still_life`.

Além disso, `abstract_painting` aparece no conjunto de teste exclusivamente em Cubism, embora com apenas 18 exemplos.

Esses resultados mostram que **gênero e macroclasse não são independentes**. Portanto, o conjunto de dados contém estrutura temática que potencialmente pode ser utilizada pelos classificadores como atalho.

O gênero `portrait` apresenta uma característica especialmente útil para os experimentos posteriores: ele está presente nas quatro macroclasses, permitindo uma avaliação parcialmente controlada do conteúdo temático.

## 10B — Desempenho dos modelos por gênero

Após identificar diferenças na composição temática das classes, o próximo passo consiste em verificar se o desempenho dos modelos também varia de acordo com o gênero representado.

São realizadas duas análises complementares:

1. **desempenho global dentro de cada gênero**, utilizando accuracy e macro-F1;
2. **desempenho condicionado simultaneamente à classe e ao gênero**, permitindo comparar, por exemplo, pinturas Renaissance religiosas com retratos Renaissance.

A segunda análise é particularmente importante, pois reduz o risco de interpretar uma accuracy elevada apenas como consequência da predominância de determinada classe dentro de um gênero.

In [ ]:
import pandas as pd

df = pd.read_csv(
    "outputs/shortcut_learning/"
    "performance_by_class_genre.csv"
)

for modelo in [
    "ResNet-50",
    "ViT-B/16",
]:
    print("\n" + "=" * 80)
    print(modelo)
    print("=" * 80)

    tabela = df[
        df["model"] == modelo
    ].copy()

    tabela["recall_within_genre"] *= 100

    display(
        tabela[
            [
                "Macro_Estilo",
                "Genero",
                "n",
                "correct",
                "recall_within_genre",
            ]
        ].round(2)
    )


ResNet-50


,Macro_Estilo,Genero,n,correct,recall_within_genre
0,Baroque,religious_painting,94,65,69.15
1,Baroque,portrait,84,68,80.95
2,Baroque,Unknown Genre,80,50,62.50
3,Baroque,genre_painting,64,48,75.00
4,Baroque,sketch_and_study,18,9,50.00
5,Baroque,still_life,13,8,61.54
6,Baroque,landscape,11,7,63.64
7,Baroque,cityscape,6,3,50.00
8,Baroque,nude_painting,3,0,0.00
9,Baroque,illustration,2,0,0.00



ViT-B/16


,Macro_Estilo,Genero,n,correct,recall_within_genre
37,Baroque,religious_painting,94,71,75.53
38,Baroque,portrait,84,72,85.71
39,Baroque,Unknown Genre,80,60,75.00
40,Baroque,genre_painting,64,58,90.62
41,Baroque,sketch_and_study,18,13,72.22
42,Baroque,still_life,13,9,69.23
43,Baroque,landscape,11,10,90.91
44,Baroque,cityscape,6,3,50.00
45,Baroque,nude_painting,3,2,66.67
46,Baroque,illustration,2,0,0.00


In [ ]:
generos_foco = [
    "religious_painting",
    "portrait",
    "landscape",
    "still_life",
]

foco = df[
    df["Genero"].isin(
        generos_foco
    )
].copy()

foco["recall_within_genre"] *= 100

display(
    foco[
        [
            "model",
            "Macro_Estilo",
            "Genero",
            "n",
            "correct",
            "recall_within_genre",
        ]
    ].round(2)
)

,model,Macro_Estilo,Genero,n,correct,recall_within_genre
0,ResNet-50,Baroque,religious_painting,94,65,69.15
1,ResNet-50,Baroque,portrait,84,68,80.95
5,ResNet-50,Baroque,still_life,13,8,61.54
6,ResNet-50,Baroque,landscape,11,7,63.64
10,ResNet-50,Cubism,still_life,99,99,100.00
13,ResNet-50,Cubism,portrait,47,44,93.62
14,ResNet-50,Cubism,landscape,29,25,86.21
19,ResNet-50,Cubism,religious_painting,2,2,100.00
21,ResNet-50,Impressionism,landscape,127,120,94.49
23,ResNet-50,Impressionism,portrait,52,45,86.54


### Resultado — desempenho condicionado ao gênero

Foram observadas accuracies elevadas em gêneros fortemente associados a determinadas classes. Entretanto, a análise condicionada simultaneamente à classe e ao gênero revelou diferenças mais informativas.

O principal caso ocorreu em **Renaissance**:

| Gênero | Recall ResNet-50 | Recall ViT-B/16 |
|---|---:|---:|
| religious_painting | 81,69% | 91,55% |
| portrait | 47,06% | 61,76% |

Ao manter a classe Renaissance fixa e alterar o gênero de `religious_painting` para `portrait`, o recall caiu:

- **34,63 pontos percentuais** na ResNet-50;
- **29,79 pontos percentuais** no ViT-B/16.

Em Baroque ocorreu comportamento diferente: retratos foram reconhecidos com maior facilidade do que pinturas religiosas.

Impressionism apresentou desempenho especialmente elevado em `landscape`, enquanto Cubism permaneceu altamente reconhecível tanto em `still_life` quanto em `portrait`.

Os resultados indicam que **o efeito do gênero não é uniforme entre as classes**, sendo Renaissance particularmente sensível à alteração do conteúdo temático.

## 10C — Baseline utilizando exclusivamente `genre`

Para medir diretamente quanta informação sobre a macroclasse está presente no gênero temático, é construído um baseline que **não utiliza os pixels das imagens**.

A regra é aprendida exclusivamente no conjunto de treino:

> para cada gênero, selecionar a macroclasse mais frequente entre as pinturas daquele gênero.

Em seguida, essa associação é aplicada ao conjunto de teste.

Exemplo conceitual:

`religious_painting → classe mais frequente entre pinturas religiosas no treino`

O conjunto de teste não participa da construção da regra.

Como existem quatro classes balanceadas, uma classificação aleatória uniforme possui referência de aproximadamente **25% de accuracy**.

Se o baseline baseado apenas em gênero superar significativamente esse valor, isso indica que o conteúdo temático contém informação relevante sobre a macroclasse.

In [ ]:
!python -m src.evaluation.genre_baseline


=== Etapa 10C — baseline genre-only ===

Regra aprendida exclusivamente no conjunto de treino:
            Genero predicted_class  predicted_idx  n_class  n_genre_train  class_share_train  n_classes_tied
     Unknown Genre         Baroque              1      379           1147              33.04               1
 abstract_painting          Cubism              3       86             86             100.00               1
         cityscape   Impressionism              2      210            341              61.58               1
    genre_painting   Impressionism              2      354            927              38.19               1
      illustration         Baroque              1        3              7              42.86               2
         landscape   Impressionism              2      582            826              70.46               1
     nude_painting          Cubism              3       96            173              55.49               1
          portrait         Baroq

### Resultado — baseline `genre-only`

O baseline utilizando exclusivamente o gênero temático apresentou:

- **Accuracy:** 52,53%;
- **Macro-Precision:** 57,89%;
- **Macro-Recall:** 52,53%;
- **Macro-F1:** 53,05%.

A referência de acaso para quatro classes balanceadas é de aproximadamente **25%**.

Portanto, sem utilizar qualquer pixel das imagens, o gênero temático permitiu uma accuracy **27,53 pontos percentuais acima do acaso**.

Comparação:

| Método | Accuracy | Macro-F1 |
|---|---:|---:|
| Genre-only | 52,53% | 53,05% |
| ResNet-50 | 81,67% | 81,28% |
| ViT-B/16 | 88,13% | 88,01% |

O resultado demonstra que o gênero contém quantidade substancial de informação sobre a macroclasse e, portanto, representa uma **fonte plausível de shortcut learning**.

Entretanto, ResNet-50 e ViT-B/16 apresentam desempenho muito superior ao baseline, indicando que suas decisões não podem ser explicadas exclusivamente pelo gênero temático.

## 10D — Avaliação controlada no subconjunto `portrait`

O gênero `portrait` foi selecionado previamente como experimento de controle por estar representado nas quatro macroclasses.

O objetivo é reduzir uma das principais diferenças de conteúdo entre as classes:

> todas as pinturas avaliadas representam o mesmo gênero temático.

O subconjunto contém **251 pinturas**:

- Renaissance: 68;
- Baroque: 84;
- Impressionism: 52;
- Cubism: 47.

Como o subconjunto não é perfeitamente balanceado, a avaliação utiliza:

- accuracy;
- balanced accuracy;
- macro-F1;
- precision, recall e F1 por classe.

Nenhuma pintura é removida para balancear artificialmente o subconjunto.

In [ ]:
!python -m src.evaluation.portrait_control \
  --resnet-predictions "/content/drive/MyDrive/TCC_WikiArt_Compartilhado/outputs/evaluation/resnet50/test_painting_predictions.csv.gz" \
  --vit-predictions "/content/drive/MyDrive/TCC_WikiArt_Compartilhado/outputs/evaluation/vit_b16/test_painting_predictions.csv.gz"


=== Etapa 10D — subconjunto controlado portrait ===

Total de pinturas: 251

Distribuição por classe:
        class  n
  Renaissance 68
      Baroque 84
Impressionism 52
       Cubism 47

=== Comparação teste completo vs portrait ===
    model  full_accuracy  portrait_accuracy  delta_accuracy  full_balanced_accuracy  portrait_balanced_accuracy  delta_balanced_accuracy  full_macro_f1  portrait_macro_f1  delta_macro_f1
ResNet-50          81.67              75.30           -6.37                   81.67                       77.04                    -4.62          81.28              75.22           -6.06
 ViT-B/16          88.13              82.47           -5.66                   88.13                       83.99                    -4.15          88.01              83.06           -4.96

=== Métricas por classe no subconjunto portrait ===
    model    genre         class  precision  recall    f1  support
ResNet-50 portrait   Renaissance      74.42   47.06 57.66       68
ResNet-50 portrai

### Resultado — controle por `portrait`

A restrição da avaliação ao gênero `portrait` reduziu o desempenho dos dois modelos.

#### ResNet-50

- Accuracy: **81,67% → 75,30%**;
- Balanced accuracy: **81,67% → 77,04%**;
- Macro-F1: **81,28% → 75,22%**.

#### ViT-B/16

- Accuracy: **88,13% → 82,47%**;
- Balanced accuracy: **88,13% → 83,99%**;
- Macro-F1: **88,01% → 83,06%**.

O maior efeito ocorreu novamente em Renaissance.

O recall da classe caiu:

- ResNet-50: **71,73% → 47,06%**;
- ViT-B/16: **82,13% → 61,76%**.

Baroque apresentou comportamento contrário, com aumento de recall no subconjunto de retratos.

Os modelos, entretanto, continuam apresentando desempenho muito superior ao acaso mesmo quando o gênero é controlado.

Esse resultado sugere que **diferenças temáticas facilitam parte da classificação, mas não são suficientes para explicar o desempenho das redes**.

## 10E — Confusões temáticas entre Renaissance e Baroque

As matrizes de confusão da Etapa 9 mostraram que Renaissance e Baroque formam o principal par de classes confundidas pelos modelos.

Nesta análise, investigamos se a direção dessas confusões muda conforme o gênero da pintura.

São comparados dois conteúdos:

- `religious_painting`;
- `portrait`.

Para cada direção é calculada a taxa:

`número de pinturas da classe X classificadas como Y / número de pinturas da classe X naquele gênero`

As duas direções são analisadas separadamente:

- Renaissance → Baroque;
- Baroque → Renaissance.

Isso permite verificar se determinada característica temática favorece sistematicamente uma das duas classes.

In [ ]:
!python -m src.evaluation.thematic_confusions \
  --resnet-predictions "/content/drive/MyDrive/TCC_WikiArt_Compartilhado/outputs/evaluation/resnet50/test_painting_predictions.csv.gz" \
  --vit-predictions "/content/drive/MyDrive/TCC_WikiArt_Compartilhado/outputs/evaluation/vit_b16/test_painting_predictions.csv.gz"


=== Etapa 10E — confusões temáticas ===

=== Renaissance ↔ Baroque ===
    model             direction              scope  n_source  n_confusions  confusion_rate  share_of_pair_errors
ResNet-50 Renaissance → Baroque          full_test       375            51           13.60                100.00
ResNet-50 Renaissance → Baroque religious_painting       213             9            4.23                 17.65
ResNet-50 Renaissance → Baroque           portrait        68            24           35.29                 47.06
ResNet-50 Baroque → Renaissance          full_test       375            73           19.47                100.00
ResNet-50 Baroque → Renaissance religious_painting        94            22           23.40                 30.14
ResNet-50 Baroque → Renaissance           portrait        84            10           11.90                 13.70
 ViT-B/16 Renaissance → Baroque          full_test       375            44           11.73                100.00
 ViT-B/16 Renaissance → 

### Resultado — efeito temático nas confusões

Foi observado um comportamento complementar nas duas direções da confusão.

#### Renaissance → Baroque

| Modelo | Religious painting | Portrait |
|---|---:|---:|
| ResNet-50 | 4,23% | 35,29% |
| ViT-B/16 | 3,76% | 26,47% |

Pinturas Renaissance religiosas são raramente confundidas com Baroque. Entretanto, quando Renaissance aparece como retrato, a taxa de confusão aumenta fortemente.

#### Baroque → Renaissance

| Modelo | Religious painting | Portrait |
|---|---:|---:|
| ResNet-50 | 23,40% | 11,90% |
| ViT-B/16 | 21,28% | 11,90% |

No sentido contrário, pinturas Baroque religiosas apresentam maior probabilidade de serem classificadas como Renaissance.

O comportamento é consistente nos dois modelos.

A associação observada no dataset:

`religious_painting → Renaissance`

parece, portanto, favorecer previsões como Renaissance.

Quando essa pista está presente em uma obra Renaissance, a confusão com Baroque diminui. Quando aparece em uma obra Baroque, a confusão com Renaissance aumenta.

Esse padrão é **compatível com o uso de pistas temáticas correlacionadas**, embora não constitua prova causal de que elementos religiosos sejam diretamente utilizados nas decisões das redes.

In [25]:
!git status


On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	outputs/shortcut_learning/

nothing added to commit but untracked files present (use "git add" to track)
